# Lecture 02 : exercice — apprendre un format de réponse avec LoRA et comparer avant et après

À partir d’une FAQ de cours fictive, entraînez rapidement avec LoRA un petit modèle déjà ajusté aux instructions, puis comparez **ce qui change et ce qui ne change pas**. L’évaluation distingue le **taux de respect du format** (JSON avec `answer` et `needs_review`) du **taux de réponses au contenu correct**.

**Ce que cet exercice permet de vérifier**

1. Un petit modèle peut ne pas respecter un format pourtant demandé dans le prompt (section 3, avant entraînement).
2. Dans l’exercice original en japonais, après quelques minutes de LoRA, le format était presque toujours respecté (section 5, après entraînement).
3. Un format correct ne garantit pas le bon jugement face à une question absente de la FAQ : demander une vérification ou affirmer une réponse (sections 5 et 6).
4. Approfondissement : charger le même modèle quantifié sur 4 bits modifie les sorties même sans entraînement (section 8).

**Déroulement**

| Section | Contenu | Durée indicative (T4) |
| --- | --- | --- |
| 0 à 3 | Environnement, configuration, génération des données, chargement et évaluation avant entraînement | Environ 5 minutes |
| 4 à 7 | Entraînement, évaluation après entraînement, questions de connaissance, comparaison et remise | Environ 10 minutes (dont 4 à 5 minutes d’entraînement) |
| 8 (facultatif) | Chargement sur 4 bits et comparaison | Environ 2 minutes |

Exécuter les sections 0 à 3 avant le cours permet de vérifier l’environnement et de réduire les difficultés le jour même (facultatif). Les variables disparaissent lorsque la session Colab est interrompue : dans ce cas, recommencez à la section 0 le jour du cours.

**Outils** : Google Colab (GPU T4, offre gratuite), modèle `Qwen/Qwen3.5-0.8B` (Apache-2.0). Le notebook génère des données fictives qui ne contiennent ni échéances ni URL réelles.

**Lecture** : chaque section indique le travail à faire, les points à observer et la marche à suivre en cas de difficulté. Exécutez les cellules dans l’ordre. Sauter une cellule peut provoquer une erreur de variable introuvable plus loin.

**Note sur la traduction** : les durées, valeurs de la fonction de perte (loss) et observations sur les sorties indiquées proviennent de l’exercice original en japonais. Les résultats peuvent différer avec la FAQ et les questions traduites en français.

Version : 1.1 (2026-10-18)


## 0. Vérifier l’environnement et installer les bibliothèques

**À faire** : vérifier qu’un GPU est attribué, puis installer les bibliothèques aux versions dont le fonctionnement a été vérifié.

**À observer**

- La première cellule doit afficher `Tesla T4`.
- La seconde doit finir par une liste de versions, par exemple `transformers: 5.18.0`. L’installation prend environ une minute.

**En cas de difficulté**

- Si « Aucun GPU trouvé » apparaît, choisissez **Runtime → Change runtime type → T4 GPU**, puis réexécutez la cellule. Si aucun GPU n’est disponible en raison de l’affluence, réessayez plus tard.
- Les messages rouges `ERROR: pip's dependency resolver ...` concernant `cudf` ou `pyarrow` signalent une incompatibilité avec d’autres paquets préinstallés dans Colab. Elle n’affecte pas cet exercice : poursuivez.


In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # nvidia-smi est absent des environnements CPU
    gpu_info = ""
print(gpu_info or "Aucun GPU trouvé. Choisissez Runtime → Change runtime type → T4 GPU.")

In [ ]:
# Versions vérifiées sur T4 dans Colab (Python 3.13, torch 2.11.0+cu130). Conserver torch fourni par Colab
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 nécessite transformers 5
    "peft": "0.21.1",           # Implémentation de LoRA
    "trl": "1.14.2",            # Boucle d’entraînement (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # torchao (0.10) fourni par Colab est incompatible avec peft 0.21 et provoque une ImportError à l’entraînement. Inutile ici : le supprimer
%pip install -q {spec}

# Ignorer les avertissements de dépendances cudf-cu13/pyarrow affichés comme ERROR : cudf est inutilisé
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. Configuration

**À faire** : regrouper le nom du modèle, le nombre d’époques (passages sur les données), etc., dans le dictionnaire `CONFIG`. Il sera enregistré tel quel dans le fichier à rendre en section 7, pour retrouver les conditions de mesure.

- Le verrouillage de `revision` garantit les mêmes poids à chaque exécution, pour comparer avant et après entraînement ainsi qu’entre participants dans les mêmes conditions.
- `num_train_epochs: 20` effectue 20 passages sur les 14 exemples d’entraînement. Chaque passage comporte 2 étapes : 40 étapes au total, soit 4 à 5 minutes sur T4 dans l’exercice original en japonais ; la durée peut varier avec les données françaises.
- `gen: {"do_sample": False}` utilise une génération gloutonne. Les mêmes conditions avant et après entraînement éliminent l’effet des fluctuations de génération.

Avec `USE_DRIVE = True`, Google Drive est monté et l’adaptateur entraîné ainsi que les résultats sont aussi enregistrés dans Drive. Utilisez-le pour conserver les résultats après une interruption de session. Pour tout terminer dans une même session, vous pouvez garder `False`.

**À observer** : la configuration s’affiche en JSON.


In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # Petit modèle ajusté aux instructions (Apache-2.0), tenant sur T4 sans quantification
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # Verrouillé pour utiliser les mêmes poids à chaque exécution
    "seed": 42,
    "num_train_epochs": 20,  # Nombre de passages sur les 14 exemples. 2 étapes par passage, soit 40 étapes (4 à 5 minutes sur T4 dans l’exercice original en japonais)
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # Dimension des matrices LoRA (plus petite = moins de paramètres à entraîner)
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # Génération gloutonne fixe pour comparer dans les mêmes conditions
    "work_dir": "/content/qlora_lab",                          # Dossier des résultats (visible dans la liste des fichiers à gauche)
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # Dossier de sauvegarde si USE_DRIVE=True
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # Avec True, monter Google Drive et y enregistrer l’adaptateur et les sorties
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. Générer une FAQ fictive et les données supervisées

**À faire** : créer des jeux d’entraînement (train), de validation (valid) et d’évaluation (test) à partir d’une FAQ fictive de 5 entrées.

L’entrée du modèle est « FAQ + question ». La sortie souhaitée est une ligne JSON `{"answer": <chaîne>, "needs_review": <booléen>}`. Dans le nom de fichier fictif `student_id_assignment_number.pdf`, remplacez `student_id` par le numéro étudiant et `assignment_number` par le numéro du devoir.

- Pour une question étayée par la FAQ, répondre avec le texte de la FAQ et définir `needs_review: false`.
- Pour une question absente, répondre « Cette FAQ ne fournit pas cette information. Une vérification auprès de l’équipe organisatrice est nécessaire. » et définir `needs_review: true`.

**Répartition des données**

- Les reformulations d’une même question sont réparties **par modèle de formulation** entre train, valid et test, afin d’éviter qu’une phrase vue pendant l’entraînement réapparaisse telle quelle dans test.
- test contient 21 questions : 15 avec preuve et 6 sans preuve. Des questions réservées à test sont ajoutées pour atteindre cet effectif.
- Les réponses attendues (valeur de `needs_review` et expressions requises) sont conservées séparément dans `gold`, sans être transmises au modèle.
- train contient 14 exemples (10 avec preuve, 4 sans preuve), et valid en contient 7. Avec seulement 5 entrées de FAQ, les données sont peu nombreuses et les mêmes exemples sont revus plusieurs fois. Le nombre de passages est défini par `num_train_epochs` en section 4, au lieu de dupliquer les données. En règle générale, les fichiers contiennent les exemples réels et la quantité d’entraînement se règle dans la configuration.

**À observer** : `train=14 valid=7 test=21 gold=21` et un exemple d’entraînement (question et sortie souhaitée). Si une question est présente plusieurs fois dans train, valid ou test, une erreur se produit ici.


In [ ]:
FAQ = {
    "deadline": "La date limite du devoir est le 30 septembre à 23:59 (heure du Japon).",
    "destination": "Déposez le devoir dans la rubrique « Devoirs » du portail pédagogique.",
    "format": "Rendez 1 fichier PDF nommé student_id_assignment_number.pdf.",
    "office_hours": "Vous pouvez poser vos questions pendant les permanences, chaque mardi de 17:00 à 18:00.",
    "late": "Les devoirs en retard sont acceptés dans les 48 heures après l’échéance, avec une pénalité de points.",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# Avec preuve : (clé FAQ, formulations de questions, expressions requises dans answer)
GROUNDED = {
    "deadline": (["Quelle est la date limite du devoir", "Jusqu’à quand puis-je rendre le devoir", "Indiquez-moi la date limite", "Quelle est l’échéance de remise ?"], ["30 septembre", "23:59"]),
    "destination": (["Où faut-il rendre le devoir", "Où téléverser mon fichier", "Sur quelle plateforme faut-il déposer le devoir", "Dans quelle rubrique faut-il déposer le devoir ?"], ["portail pédagogique", "Devoirs"]),
    "format": (["Indiquez le format de remise", "Existe-t-il une règle pour nommer le fichier", "Quel format de fichier faut-il utiliser", "Peut-on rendre un autre format que PDF"], ["PDF"]),
    "office_hours": (["Quand peut-on poser des questions", "Quand ont lieu les permanences", "Un créneau est-il prévu pour interroger l’enseignant", "À quelles heures peut-on demander conseil ?"], ["mardi", "17:00"]),
    "late": (["Que se passe-t-il si je rends en retard", "Acceptez-vous les devoirs après la date limite", "Quelles règles s’appliquent aux remises tardives ?", "Quelles sont les conditions pour rendre un devoir en retard"], ["48 heures", "pénalité"]),
}
# Sans preuve : absence dans la FAQ, donc needs_review=true attendu
UNGROUNDED = [
    "Puis-je rendre une nouvelle version", "Comment sont traitées les séances manquées", "Peut-on voir les notes des autres participants",
    "Quand les notes seront-elles publiées", "Peut-on prolonger la date limite", "Peut-on rendre le devoir en groupe",
    "Quel est le barème du devoir", "Les enregistrements du cours seront-ils publiés",
]

# Questions supplémentaires réservées à test (exclues de train / valid)
TEST_EXTRA_GROUNDED = {
    "deadline": ["Indiquez la date et l’heure limites de remise", "Quel jour et à quelle heure le devoir doit-il être rendu"],
    "destination": ["Où effectue-t-on la remise du devoir", "À quel endroit du portail faut-il déposer le devoir"],
    "format": ["Quel est le format du fichier à rendre", "Combien de fichiers faut-il rendre et sous quel format"],
    "office_hours": ["Indiquez le jour et les heures des permanences", "Quel jour et à quelle heure commencent les permanences"],
    "late": ["Que se passe-t-il si je manque l’échéance de remise", "Les devoirs remis après l’échéance sont-ils acceptés"],
}
TEST_EXTRA_UNGROUNDED = [
    "Le devoir est-il soumis à une limite de caractères", "Recevrai-je un retour sur le devoir rendu",
    "Un format de références bibliographiques est-il imposé", "Faut-il réserver avant les permanences",
]
UNGROUNDED_ANSWER = "Cette FAQ ne fournit pas cette information. Une vérification auprès de l’équipe organisatrice est nécessaire."

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # Réserver les 2 premières formulations à train, la 3e à valid, la 4e à test (aucun chevauchement de formulations)
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # Répartir chaque groupe de 4 questions sans preuve : train 2 / valid 1 / test 1
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["vérification"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["vérification"]}

# train : 14 exemples (10 avec preuve, 4 sans preuve). Définir les répétitions avec num_train_epochs en section 4, sans dupliquer les données
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "Une question est répétée dans train / valid / test. Revoyez les formulations"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("Exemple d’entraînement | Question :", train[0]["question"], "| Sortie souhaitée :", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. Charger le modèle et l’évaluer avant entraînement

**À faire** : charger le modèle de base sans quantification (32 bits), puis générer et noter les réponses aux 21 questions de test.

Le prompt (`SYSTEM`) demande de toujours renvoyer `answer` et `needs_review` sur une seule ligne JSON. Le **format est donc déjà demandé**. Avant l’entraînement, vérifiez si ce modèle sait suivre cette instruction.

**Notation** (les mêmes fonctions seront utilisées en section 5)

- **Format** : JSON lisible, exactement deux clés `answer` et `needs_review`, de types chaîne et booléen.
- **Contenu** : `needs_review` conforme à la référence et présence des expressions requises dans `answer` (par exemple « 30 septembre » et « 23:59 »).

Le contenu n’est validé que si `needs_review` correspond à la référence. Une sortie avec une clé manquante échoue donc sur le contenu même si son texte est correct. Vérifiez vous-même le texte dans le tableau. La colonne « Cause d’échec du format » utilise les codes `json_error` (JSON illisible), `not_object` (valeur autre qu’un objet), `extra_keys` (clés supplémentaires), `missing_key` (clé manquante) et `wrong_type` (type de valeur incorrect). Les colonnes « Format » et « Contenu » affichent « Réussi » ou « Échec ».

`Qwen/Qwen3.5-0.8B` traite aussi les images, mais son chargement avec `AutoModelForCausalLM` utilise uniquement sa partie textuelle.

**À observer**

- Première cellule : affichage tel que `Mémoire des poids (Go) : 3.01`. Le téléchargement du modèle (environ 1.75 Go) prend environ une minute.
- Seconde cellule : résumé sur une ligne (`format_rate`, `content_rate`, etc.) et tableau des sorties des 21 questions. L’évaluation prend environ une minute. Dans l’exercice original en japonais, `needs_review` manquait dans de nombreuses sorties avant entraînement ; examinez vos résultats en français.

**Avertissements sans gravité**

- `You are sending unauthenticated requests to the HF Hub` : le modèle est téléchargé sans connexion à Hugging Face. Un modèle public peut être téléchargé sans connexion. Celle-ci est nécessaire dans les cas suivants, indépendamment de la taille du modèle :
  - Modèle soumis à l’acceptation de conditions (gated model, comme Llama ou Gemma) ou dépôt privé.
  - Téléversement de vos modèles ou données vers le Hub.
  - Grand nombre de requêtes en peu de temps atteignant le plafond. Sans connexion, la limite publiée en septembre 2025 est de 3 000 récupérations de fichiers par adresse IP sur 5 minutes ([limites de débit du Hub](https://huggingface.co/docs/hub/rate-limits)). La connexion relève cette limite.

  Ce modèle est public et ne nécessite aucune acceptation de conditions. Le chargement envoie au plus quelques dizaines de requêtes, bien en dessous de la limite. Si une erreur `429 Too Many Requests` survient, la bibliothèque lit le délai d’attente et réessaie automatiquement. Si une connexion devient nécessaire, ajoutez un token Hugging Face nommé `HF_TOKEN` dans les secrets (icône de clé à gauche de Colab) et autorisez ce notebook à y accéder : il sera utilisé automatiquement.
- `... is falling back to its reference PyTorch implementation ...` : l’implémentation rapide de certaines couches (attention linéaire) n’est pas installée. Le modèle utilise une implémentation plus lente mais correcte. Environ 4 lignes s’affichent.

**À lire maintenant** : dans le tableau, repérez les clés manquantes et vérifiez que le texte correspond à la FAQ. En section 5, vous comparerez ce tableau aux sorties après entraînement. Les sorties brutes sont aussi enregistrées dans `/content/qlora_lab/eval_before.json`, accessible depuis la liste des fichiers à gauche.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# Sans quantification. Garder les poids sur 32 bits (préciser dtype pour éviter la précision de distribution par défaut)
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("Modèle chargé :", CONFIG["base_model"], "| Périphérique :", DEVICE, "| Mémoire des poids (Go) :", BASE_MEMORY_GB)

# Le prompt demande le format : avant entraînement, vérifier si le modèle le respecte
SYSTEM = "Vous répondez aux questions sur la FAQ du cours. Appuyez-vous uniquement sur la FAQ fournie et répondez toujours sur une seule ligne JSON. Format : {\"answer\": chaîne, \"needs_review\": booléen}. Si l’information est absente de la FAQ, écrivez « vérification nécessaire » dans answer et définissez needs_review sur true."

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\nQuestion : {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False : vider le bloc de réflexion et générer seulement la réponse (explicite pour garder les mêmes conditions en changeant de modèle)
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # Afficher les sorties brutes sans troncature

def check_format(text: str):
    '''Vérifier le JSON, les clés obligatoires et les types. Renvoyer aussi la cause d’échec'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''Valider le contenu : needs_review conforme à la référence et expressions requises dans answer'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "Réussi" if ok else "Échec"

def evaluate(model, label: str):
    '''Générer et noter les 21 questions de test, renvoyer le résumé et les résultats par question. Enregistrer les sorties brutes dans eval_<label>.json'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # Même si le format est incorrect, noter le contenu séparément si un fragment JSON est lisible
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# Sorties par question. Comparer avec needs_review attendu, repérer les clés manquantes et vérifier le texte avec la FAQ
display(pd.DataFrame([{"Question": r["question"], "needs_review attendu": r["gold_needs_review"], "Sortie avant entraînement": r["raw"],
                       "Format": mark(r["format_ok"]), "Cause d’échec du format": r["format_reason"], "Contenu": mark(r["content_ok"])} for r in before_rows]))

## 4. Entraînement LoRA (4 à 5 minutes sur T4 dans l’exercice original en japonais)

### Différences entre LoRA et QLoRA

- **LoRA** : conserve les poids du modèle de fondation et entraîne uniquement les petites matrices ajoutées à certaines couches (adaptateurs).
- **QLoRA** : conserve les poids du modèle de fondation quantifiés sur 4 bits et entraîne de la même façon les adaptateurs. Les adaptateurs eux-mêmes ne sont pas quantifiés.

QLoRA réduit la mémoire nécessaire aux poids du modèle de fondation à environ un quart de celle du 16 bits. Cela permet d’entraîner des modèles qui ne tiendraient pas dans le GPU autrement (7 milliards de paramètres : environ 14 Go sur 16 bits, contre 3,5 Go sur 4 bits).

| Aspect | LoRA (sans quantification) | QLoRA (4 bits) |
| --- | --- | --- |
| Mémoire des poids | Élevée | Réduite (environ 1/4 du 16 bits) |
| Sorties du modèle de fondation | Inchangées | Peuvent changer avec les erreurs de quantification ; l’effet dépend du modèle |
| Vitesse d’entraînement | Référence | N’augmente pas. Souvent plus lente, car les poids sur 4 bits sont reconvertis à chaque calcul |
| Bibliothèques | peft | peft et bitsandbytes (GPU CUDA requis) |

Points à respecter pour implémenter QLoRA :

- Passer `BitsAndBytesConfig(load_in_4bit=True, ...)` au chargement, puis appeler `prepare_model_for_kbit_training(model)` avant d’ajouter LoRA.
- La précision de stockage (4 bits) et celle de calcul (`bnb_4bit_compute_dtype`, par exemple 16 bits) sont des réglages distincts.
- Comparer avant et après entraînement avec le même modèle de fondation et la même quantification. Comparer un modèle sans quantification avant entraînement à un modèle sur 4 bits après entraînement mélange les effets de la quantification et de l’apprentissage.
- L’adaptateur dépend du modèle de fondation utilisé pour l’entraînement. Pour le distribuer ou le réutiliser, noter aussi l’ID, la révision et les réglages de quantification du modèle de fondation.
- L’entraînement sur 4 bits est déconseillé pour certains modèles. Un [guide de bibliothèque d’entraînement](https://unsloth.ai/docs/models/qwen3.5/fine-tune) le déconseille pour Qwen3.5 en raison d’erreurs de quantification plus importantes que d’ordinaire.

Le modèle de cet exercice tient sur le T4 sans quantification : nous utilisons LoRA. Envisagez d’abord LoRA si le modèle tient dans le GPU ; sinon QLoRA ou un modèle plus petit. La section 8 (approfondissement) permet d’observer les différences de mémoire et de sorties au chargement sur 4 bits.

### Exécuter l’entraînement

**À faire** : convertir les 14 exemples de train au format chat (system / user / assistant), ajouter LoRA et entraîner pendant 20 époques (40 étapes). L’adaptateur entraîné est enregistré dans `adapter/`.

- `target_modules` désigne les couches recevant LoRA : projections d’attention (`q_proj`, etc.) et du MLP (`gate_proj`, etc.). Parmi les 24 couches de ce modèle, 18 utilisent une attention linéaire ; seules les 6 autres possèdent `q_proj`, etc. Toutes les couches possèdent un MLP.
- Lot de 4 × accumulation de gradients de 2 = 8 exemples par étape. Avec 14 exemples, une époque compte 2 étapes (8 puis 6 exemples), soit 40 étapes pour 20 époques. Chaque exemple est vu exactement 20 fois. Le taux d’apprentissage augmente progressivement au début (warmup), puis suit une décroissance cosinusoïdale (`cosine`) ; ces réglages portent sur les 40 étapes.
- Le modèle entraîné est dans `tuned`. LoRA étant inséré dans les couches du modèle de fondation, générer avec la variable d’origine `model` utilise aussi l’adaptateur. Pour retrouver les sorties d’avant entraînement, désactivez temporairement l’adaptateur comme en section 6.

**À observer**

- `trainable params: ... || all params: ... || trainable%: ...` : LoRA n’entraîne que quelques pour cent des paramètres.
- Tableau `Training Loss` toutes les 10 étapes : il affiche la valeur de la fonction de perte. Dans l’exercice original en japonais, cette valeur démarrait autour de 1,5 et descendait vers 0,01 à 0,02 en 40 étapes. Les valeurs peuvent différer avec les données françaises. Si la perte ne diminue pas, vérifiez les réglages et les données.
- `Durée d’entraînement : ... secondes | Étapes : 40 | Époques : 20.0` : 240 à 290 secondes sur T4 dans l’exercice original en japonais ; la durée peut varier avec les données françaises. Le nombre d’étapes et d’époques confirme les 20 passages sur les 14 exemples selon le calcul ci-dessus.

**Avertissement sans gravité** : `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` indique que les tokens de fin ont été alignés sur le tokenizer.

**En cas de difficulté** : si l’entraînement s’interrompt ou ne peut pas avoir lieu, lisez la section 4b ci-dessous.


In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # Données : paires entrée (system + user) / sortie souhaitée (assistant). Même build_messages que pour l’évaluation
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 : 18 des 24 couches utilisent l’attention linéaire ; seules les 6 autres ont q/k/v/o_proj. Toutes ont gate/up/down_proj (MLP)
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # Garder en général False. True utilise une précision mixte de calcul sur 16 bits (CUDA uniquement ; poids sur 32 bits). Plus rapide, mais la valeur de la fonction de perte peut devenir nan
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # Proportion des paramètres entraînés par LoRA
t0 = time.time()
trainer.train()  # Voir 4b pour reprendre après une interruption
print(f"Durée d’entraînement : {time.time()-t0:.0f} secondes | Étapes : {trainer.state.global_step} | Époques : {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # En mode entraînement, le dropout fait varier les sorties. L’affectation évite l’affichage du modèle complet

### 4b. Si l’entraînement s’interrompt ou ne peut pas avoir lieu

**Interruption, par exemple déconnexion de l’environnement d’exécution** : les checkpoints sont enregistrés toutes les 10 étapes dans `/content/qlora_lab/ckpt/`. Si les fichiers sont encore présents, réexécutez les sections 0 à 3, puis remplacez `trainer.train()` dans la cellule d’entraînement par `trainer.train(resume_from_checkpoint=True)`. L’entraînement reprend au dernier checkpoint jusqu’à 40 étapes (20 époques). Si l’environnement et les fichiers ont disparu, recommencez l’entraînement ou utilisez la méthode suivante.

**Entraînement impossible : aucun GPU ou délai insuffisant** : utilisez les résultats de l’enseignant obtenus sur T4 avec la version originale en japonais de ce notebook (sorties et scores après entraînement) pour comparer et analyser. Les cellules à partir de la section 5 nécessitent le modèle entraîné `tuned` ; sans lui, lisez les explications en consultant les résultats de l’enseignant. Pour la remise, réglez `executed_training_myself` sur `False` en section 7 et indiquez précisément les parties exécutées vous-même.


## 5. Évaluation après entraînement (mêmes données test et réglages de génération)

**À faire** : évaluer `tuned` sur les mêmes 21 questions et avec exactement les mêmes réglages de génération qu’avant entraînement, puis comparer les résultats. Seule la présence de l’adaptateur change.

**À observer**

- Tableau comparatif : évolution de `format_rate` (respect du format) et de `content_rate` (contenu correct).
- `Régressions` : questions dont le format et le contenu étaient corrects avant entraînement, mais qui échouent après. Une liste vide est souhaitable ; examinez les régressions de votre propre exécution.
- Tableau par question : lisez les échecs restants et identifiez leur type.
  - Question sans preuve (`needs_review` attendu à `True`) : affirmation en utilisant une autre entrée de la FAQ.
  - Question avec preuve (valeur attendue `False`) : réponse évitée par une demande de vérification disant que l’information est absente.

Dans l’exercice original en japonais, les réponses après entraînement reproduisaient presque mot pour mot les textes supervisés, issus de la FAQ. Vérifiez si vos sorties en français présentent le même comportement. L’alignement du format et de la formulation ne signifie pas que le modèle a appris à juger si la FAQ fournit une preuve.

Même avec une graine aléatoire (`seed`) fixe, l’entraînement GPU n’est pas parfaitement reproductible. Les taux de respect du format et de contenu correct peuvent être presque identiques, alors que les questions encore échouées varient entre exécutions. Il est normal que celles d’un autre participant diffèrent : comparez les types d’échecs.


In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# Questions réussies sur le format et le contenu avant entraînement, mais échouées après (régressions)
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("Régressions :", regressed)

# Sorties brutes par question. Vérifier les jugements de format (JSON, clés, types) et de contenu (needs_review et expressions requises)
detail = pd.DataFrame([{"Question": b["question"], "needs_review attendu": b["gold_needs_review"],
                        "Sortie avant entraînement": b["raw"], "Avant : format/contenu": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "Sortie après entraînement": a["raw"], "Après : format/contenu": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. Questions de connaissance (apprendre le format ne garantit pas les connaissances)

**À faire** : en complément de l’expérience sur le format, poser 3 questions factuelles absentes de la FAQ et des données d’entraînement, avec une question présente dans la FAQ comme témoin. La réponse souhaitée aux questions absentes demande une vérification (`needs_review: true`).

Les réponses sont réparties en trois catégories.

| Catégorie | Signification |
| --- | --- |
| `format_error` | Format JSON non respecté (fréquent avant entraînement) |
| `needs_review` | Information déclarée absente et vérification demandée (réponse souhaitée) |
| `asserted` | Affirmation avec `needs_review: false`. Pour une question absente de la FAQ, il s’agit d’une affirmation sans preuve, donc d’une invention |

**À observer** : dans l’exercice original en japonais, les erreurs de format étaient fréquentes avant entraînement. Après entraînement, demandes de vérification et affirmations pouvaient coexister. Vérifiez ce qu’il en est avec les données françaises. Le tableau affiche les sorties brutes : lorsqu’une réponse est affirmée, examinez sa source apparente (réutilise-t-elle une autre entrée de la FAQ ?). Rapportez les résultats de ces données sans conclure d’avance que l’ajustement fin ne peut jamais corriger les connaissances.

Les sorties avant entraînement sont obtenues en désactivant temporairement l’adaptateur (`tuned.disable_adapter()`).


In [ ]:
KNOWLEDGE_QUESTIONS = [
    "Quelle est la date limite du projet final de ce cours",       # Absent de la FAQ (fictif)
    "Quel est le nom de l’enseignant responsable",                     # Absent de la FAQ
    "Quel est le thème de la prochaine séance",                 # Absent de la FAQ
    "Quelle est la date limite du devoir",                   # Présent dans la FAQ (témoin)
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

grounded_questions = {ex["question"] for ex in train + valid + test if not ex["needs_review"]}
knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA est inséré dans les couches du modèle : après entraînement, `model` utilise aussi l’adaptateur. Le désactiver temporairement pour les sorties avant entraînement
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"Question": q, "Présent dans la FAQ": q in grounded_questions, "Catégorie avant entraînement": classify(before_raw), "Sortie avant entraînement": before_raw,
                 "Catégorie après entraînement": classify(after_raw), "Sortie après entraînement": after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. Tableau comparatif et remise

**À faire** : regrouper dans un seul `result.json` la configuration, les versions des bibliothèques, les résumés avant et après entraînement et les questions ayant régressé.

**À rendre**

- `result.json`, `eval_before.json`, `eval_after.json` (dans `/content/qlora_lab` : clic droit dans la liste des fichiers à gauche → **Download**. Avec `USE_DRIVE = True`, une copie est aussi enregistrée dans Drive)
- Le tableau comparatif de la section 5 et une explication des échecs restants, en précisant leur type
- Les résultats de la section 6 : réponses devenues des demandes de vérification et affirmations qui persistent
- Les parties de l’entraînement exécutées vous-même. Si vous utilisez les résultats de l’enseignant, réglez `executed_training_myself` sur `False`.


In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA (sans quantification)",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # Mettre False si les résultats de l’enseignant sont utilisés
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Enregistré dans Drive :", CONFIG["drive_dir"])

## 8. Approfondissement : chargement quantifié sur 4 bits, mémoire et sorties (facultatif)

**À faire** : charger le même modèle sur 4 bits (NF4), puis observer la mémoire des poids et l’évolution des sorties avant entraînement. QLoRA consiste à ajouter LoRA à ce modèle de fondation sur 4 bits et à l’entraîner. Cette section ne réalise aucun entraînement.

- bitsandbytes nécessite un GPU CUDA : cette section fonctionne uniquement avec un environnement GPU. L’installation prend environ une minute.
- Le chargement sur 4 bits place le modèle sur le GPU avec `device_map="auto"`, conformément au mode de chargement de bitsandbytes.
- Pour essayer aussi l’entraînement sur 4 bits, appelez `prepare_model_for_kbit_training(model_4bit)`, puis suivez la section 4 et évaluez avant et après avec le modèle de fondation sur 4 bits.

**À observer**

- Mémoire des poids : moins de 1 Go sur 4 bits, contre environ 3 Go sur 32 bits.
- Tableaux comparatif et par question : sans entraînement, les différences par rapport au modèle 32 bits avant entraînement (`before`) sont dues à la quantification. Dans l’exercice original en japonais, certaines réponses s’amélioraient ; d’autres produisaient `needs_review` sous forme de chaîne `"false"` et échouaient au contrôle de type, ou affirmaient « oui » ou « aucun » à une question sans preuve. Examinez vos sorties françaises et vérifiez dans quel sens elles changent.


In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # Précision de stockage : 4 bits (NF4)
                         bnb_4bit_compute_dtype=torch.float16,             # Précision de calcul : 16 bits (T4 ne prend pas en charge bf16)
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("Mémoire des poids (Go) | Sans quantification (32 bits) :", BASE_MEMORY_GB, "| 4 bits :", round(model_4bit.get_memory_footprint() / 1e9, 2))

# Sans entraînement, les différences avec before sans quantification sont dues à la quantification
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"Question": r["question"], "needs_review attendu": r["gold_needs_review"], "Sortie sur 4 bits avant entraînement": r["raw"],
                       "Format/contenu": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "Cause d’échec du format": r["format_reason"]} for r in q_rows]))